# Spatial interpolation lab
Estimate values between observations, then inspect how the assumptions differ between inverse distance weighting and ordinary kriging.

All coordinates are synthetic planar units. For real data, choose and document a projected CRS and use spatially appropriate validation.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/geostatistics'))
from interpolation import idw_predict, ordinary_kriging_predict


## 1. Inspect the observations

The sample values follow a broad spatial trend with a local variation. The sample pattern is intentionally irregular: interpolation is influenced by both the values and where observations were collected.

In [ ]:
samples = [(0, 0, 8), (2, 1, 11), (5, 0, 13), (8, 2, 18), (1, 5, 14), (4, 4, 19), (7, 5, 22), (2, 8, 18), (6, 9, 25), (9, 8, 27)]
xy = np.asarray([(x, y) for x, y, _ in samples])
z = np.asarray([value for _, _, value in samples])
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(xy[:, 0], xy[:, 1], c=z, s=90, edgecolor='black')
for i, (x, y) in enumerate(xy): ax.annotate(f'{z[i]:.0f}', (x, y), xytext=(5, 5), textcoords='offset points')
ax.set_title('Observed sample values'); ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_aspect('equal'); ax.grid(True)
fig.colorbar(sc, ax=ax, label='Observed value'); plt.show()


## 2. Inverse distance weighting

IDW is a deterministic weighted average. Change the power and observe how strongly the nearest samples dominate. It does not estimate a stochastic prediction variance.

In [ ]:
grid_x = np.linspace(0, 9, 70)
grid_y = np.linspace(0, 9, 70)
GX, GY = np.meshgrid(grid_x, grid_y)
targets = list(zip(GX.ravel(), GY.ravel()))
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
for ax, power in zip(axes, [1, 2, 4]):
    pred = np.asarray(idw_predict(samples, targets, power=power)).reshape(GX.shape)
    im = ax.pcolormesh(GX, GY, pred, shading='auto')
    ax.scatter(xy[:, 0], xy[:, 1], c=z, edgecolor='black', s=35)
    ax.set_title(f'IDW power = {power}'); ax.set_aspect('equal'); ax.set_xlabel('x'); ax.set_ylabel('y')
    fig.colorbar(im, ax=ax, label='Estimated value')
plt.show()


## 3. Ordinary kriging with an explicit variogram

The model below is an isotropic exponential semivariogram with zero nugget. The parameters are deliberately supplied rather than fitted. The estimate and variance are conditional on this model.

In [ ]:
kriged = ordinary_kriging_predict(samples, targets, partial_sill=90, range_scale=4.5)
estimates = np.asarray([item[0] for item in kriged]).reshape(GX.shape)
variances = np.asarray([item[1] for item in kriged]).reshape(GX.shape)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, field, title in [(axes[0], estimates, 'Ordinary kriging estimate'), (axes[1], variances, 'Model-based kriging variance')]:
    im = ax.pcolormesh(GX, GY, field, shading='auto')
    ax.scatter(xy[:, 0], xy[:, 1], c='white', edgecolor='black', s=30)
    ax.set_title(title); ax.set_aspect('equal'); ax.set_xlabel('x'); ax.set_ylabel('y')
    fig.colorbar(im, ax=ax)
plt.show()
print('Estimate at centre and model variance:', ordinary_kriging_predict(samples, [(4.5, 4.5)], partial_sill=90, range_scale=4.5)[0])


## 4. A spatial holdout is a different question from a random split

Hold out the eastern observations, predict them from the western observations and calculate errors. This is a simple spatially separated illustration, not a universal cross-validation design. It may be challenging because the held-out locations extend beyond the training cluster.

In [ ]:
train = [sample for sample in samples if sample[0] <= 6]
test = [sample for sample in samples if sample[0] > 6]
test_xy = [(x, y) for x, y, _ in test]
idw_test = idw_predict(train, test_xy, power=2)
kriging_test = ordinary_kriging_predict(train, test_xy, partial_sill=90, range_scale=4.5)
actual = np.asarray([value for _, _, value in test])
idw_rmse = np.sqrt(np.mean((np.asarray(idw_test) - actual) ** 2))
kriging_rmse = np.sqrt(np.mean((np.asarray([p[0] for p in kriging_test]) - actual) ** 2))
print('Held-out points:', len(test))
print('IDW RMSE:', round(float(idw_rmse), 3))
print('Ordinary kriging RMSE:', round(float(kriging_rmse), 3))
print('Kriging variances at held-out points:', [round(p[1], 3) for p in kriging_test])


## 5. Investigate

1. Why does increasing IDW power make the surface more locally influenced?
2. Change the kriging range scale. What changes in the estimate and variance?
3. What does the kriging variance describe, and which sources of uncertainty does this implementation omit?
4. Why might a random train/test split overstate performance for spatially autocorrelated observations?
5. Which CRS and distance units would be appropriate for a real study area?
